In [4]:
import openai
import os 
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client

In [5]:
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

Download all data from qdrant

In [6]:
all_points = qdrant_client.scroll(
    collection_name="amazon_reviews_collection",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [7]:
all_points

([Record(id=0, payload={'product_id': 'B002', 'product_name': 'Stainless Steel Water Bottle', 'review_text': 'This product has arrived on time. amazing quality.', 'rating': 2, 'details': {'Date First Available': 'August 26, 2024', 'Manufacturer': 'Brand A', 'ASIN': 'ASIN5114298'}, 'helpful_votes': 43}, vector=None, shard_key=None, order_value=None),
  Record(id=1, payload={'product_id': 'B006', 'product_name': 'Yoga Mat', 'review_text': 'This product has feels premium. solid choice.', 'rating': 4, 'details': {'Date First Available': 'November 11, 2024', 'Manufacturer': 'Brand B', 'ASIN': 'ASIN7115185'}, 'helpful_votes': 27}, vector=None, shard_key=None, order_value=None),
  Record(id=2, payload={'product_id': 'B009', 'product_name': 'Backpack', 'review_text': 'This product has customer support was helpful. amazing quality.', 'rating': 4, 'details': {'Date First Available': 'March 25, 2024', 'Manufacturer': 'Brand D', 'ASIN': 'ASIN9467237'}, 'helpful_votes': 46}, vector=None, shard_key=

In [8]:
all_points[0][0].payload

{'product_id': 'B002',
 'product_name': 'Stainless Steel Water Bottle',
 'review_text': 'This product has arrived on time. amazing quality.',
 'rating': 2,
 'details': {'Date First Available': 'August 26, 2024',
  'Manufacturer': 'Brand A',
  'ASIN': 'ASIN5114298'},
 'helpful_votes': 43}

In [9]:
all_context = [
	{
		"id": payload.get("ASIN") or payload.get("product_id"),
		"review_text": payload.get("review_text"),
	}
	for data in all_points[0]
	if (payload := data.payload) is not None
]

In [10]:
all_context

[{'id': 'B002',
  'review_text': 'This product has arrived on time. amazing quality.'},
 {'id': 'B006',
  'review_text': 'This product has feels premium. solid choice.'},
 {'id': 'B009',
  'review_text': 'This product has customer support was helpful. amazing quality.'},
 {'id': 'B004',
  'review_text': 'This product has packaging was neat. pretty decent.'},
 {'id': 'B002',
  'review_text': 'This product has arrived on time. could be better.'},
 {'id': 'B009',
  'review_text': 'This product has battery life could be better. amazing quality.'},
 {'id': 'B006',
  'review_text': 'This product has easy to use. satisfied with purchase.'},
 {'id': 'B003',
  'review_text': 'This product has arrived on time. not bad at all.'},
 {'id': 'B004',
  'review_text': 'This product has good value for money. could be better.'},
 {'id': 'B001',
  'review_text': 'This product has highly recommended. solid choice.'},
 {'id': 'B009',
  'review_text': 'This product has packaging was neat. could be better.'},

Render a prompt to generate synthetic Eval Reference Dataset

In [11]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context chunks.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the retrieved chunks.",
            }
        }
    }
}

In [12]:
SYSTEM_PROMPT = """You are a helpful assistant for generating evaluation data for a retrieval-augmented generation (RAG) system.
Your task is to generate a set of questions that can be answered using the provided context chunks, which are reviews of products. For each question, you should also provide:
1. The IDs of the context chunks that could be used to answer the question.
2. An example answer grounded in the context chunks.
3. A reasoning explanation for why the question can be answered with the retrieved chunks.  
Make sure the questions are relevant to the content of the reviews and that the example answers are accurate and based on the information in the context chunks. The reasoning should clearly explain the connection between the question, the context chunks, and the answer.
4. Construct 5 questions with multiple choice answers, where each question has 4 options (A, B, C, D) and only one correct answer. The options should be plausible and based on the context chunks, but only one should be correct.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>
I need to be able to parse the json output
"""

USER_PROMPT = f"""Here is the lost of chunks, each list element is a dictionary with id and text:
{all_context}
Please generate the evaluation data as per the system prompt instructions.
"""

In [13]:
print(SYSTEM_PROMPT)

You are a helpful assistant for generating evaluation data for a retrieval-augmented generation (RAG) system.
Your task is to generate a set of questions that can be answered using the provided context chunks, which are reviews of products. For each question, you should also provide:
1. The IDs of the context chunks that could be used to answer the question.
2. An example answer grounded in the context chunks.
3. A reasoning explanation for why the question can be answered with the retrieved chunks.  
Make sure the questions are relevant to the content of the reviews and that the example answers are accurate and based on the information in the context chunks. The reasoning should clearly explain the connection between the question, the context chunks, and the answer.
4. Construct 5 questions with multiple choice answers, where each question has 4 options (A, B, C, D) and only one correct answer. The options should be plausible and based on the context chunks, but only one should be cor

In [14]:
print(USER_PROMPT)

Here is the lost of chunks, each list element is a dictionary with id and text:
[{'id': 'B002', 'review_text': 'This product has arrived on time. amazing quality.'}, {'id': 'B006', 'review_text': 'This product has feels premium. solid choice.'}, {'id': 'B009', 'review_text': 'This product has customer support was helpful. amazing quality.'}, {'id': 'B004', 'review_text': 'This product has packaging was neat. pretty decent.'}, {'id': 'B002', 'review_text': 'This product has arrived on time. could be better.'}, {'id': 'B009', 'review_text': 'This product has battery life could be better. amazing quality.'}, {'id': 'B006', 'review_text': 'This product has easy to use. satisfied with purchase.'}, {'id': 'B003', 'review_text': 'This product has arrived on time. not bad at all.'}, {'id': 'B004', 'review_text': 'This product has good value for money. could be better.'}, {'id': 'B001', 'review_text': 'This product has highly recommended. solid choice.'}, {'id': 'B009', 'review_text': 'This pro

In [24]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False


In [25]:
response  = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort = 'minimal'
)

print(response.choices[0].message.content)

{
  "questions": [
    {
      "id": 1,
      "question": "Which product IDs have reviews that mention 'battery life could be better'?",
      "relevant_chunk_ids": [
        "B009",
        "B006",
        "B007",
        "B005"
      ],
      "answer": "B009, B006, B007, and B005 have reviews mentioning 'battery life could be better.'",
      "reasoning": "The provided chunks include explicit review_text strings that contain the phrase 'battery life could be better.' Specifically: B009 appears multiple times with 'battery life could be better' (e.g. 'This product has battery life could be better. amazing quality.' and '... great product.'), B006 has 'This product has battery life could be better. amazing quality.' and '... satisfied with purchase.'; B007 has 'This product has battery life could be better. satisfied with purchase.'; and B005 includes 'This product has battery life could be better. pretty decent.' Therefore these chunk IDs directly support the answer."
    },
    {
   

In [26]:
import json 
json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [27]:
json_output

{'questions': [{'id': 1,
   'question': "Which product IDs have reviews that mention 'battery life could be better'?",
   'relevant_chunk_ids': ['B009', 'B006', 'B007', 'B005'],
   'answer': "B009, B006, B007, and B005 have reviews mentioning 'battery life could be better.'",
   'reasoning': "The provided chunks include explicit review_text strings that contain the phrase 'battery life could be better.' Specifically: B009 appears multiple times with 'battery life could be better' (e.g. 'This product has battery life could be better. amazing quality.' and '... great product.'), B006 has 'This product has battery life could be better. amazing quality.' and '... satisfied with purchase.'; B007 has 'This product has battery life could be better. satisfied with purchase.'; and B005 includes 'This product has battery life could be better. pretty decent.' Therefore these chunk IDs directly support the answer."},
  {'id': 2,
   'question': "Which product IDs have at least one review that prais

In [40]:
# Safely inspect the first generated question (avoid KeyError if json_output is a dict)
if isinstance(json_output, dict):
    questions = json_output.get("questions") or json_output.get("multiple_choice_questions") or []
    if questions:
        questions[0]
    else:
        json_output
else:
    json_output

In [41]:
len(json_output)

2

In [32]:
points = qdrant_client.scroll(
    collection_name="amazon_reviews_collection",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="ASIN",
                match=MatchValue(value="B009")
            )
        ]
    ),
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)[0]

In [34]:
points

[]

In [35]:
def get_details_for_chunk_ids(chunk_ids):
    details = []
    for chunk_id in chunk_ids:
        point = qdrant_client.scroll(
            collection_name="amazon_reviews_collection",
            scroll_filter=Filter(
                must=[
                    FieldCondition(
                        key="id",
                        match=MatchValue(value=chunk_id)
                    )
                ]
            ),
            limit=1,
            offset=None,
            with_payload=True,
            with_vectors=False
        )[0]
        if point:
            details.append(point[0].payload)
    return details

In [38]:
get_details_for_chunk_ids(['B009'])

[]

Create eval dataset in langsmith

In [42]:
client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [43]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(dataset_name=dataset_name, description="Dataset for evaluating RAG system performance on question answering tasks based on product reviews.")


In [51]:
questions = []
if isinstance(json_output, dict):
    questions = json_output.get("questions", []) + json_output.get("multiple_choice_questions", [])
elif isinstance(json_output, list):
    questions = json_output

for item in questions:
    if not isinstance(item, dict):
        continue

    question = item.get("question")
    if not question:
        continue

    answer = item.get("answer") or item.get("answer_example") or item.get("answer_explanation") or "",
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": question},
        outputs={"answer": answer},
    )